# 2 - Intégration des différents composants

Ce notebook illustre un **workflow complet** en combinant l'ensemble des composants du package :
construction de la base, mise à jour, suppression, audit d'intégrité, maintenance physique et voyage dans le temps.

Il constitue le point d'entrée recommandé pour comprendre comment les classes interagissent dans un scénario de production, en partageant une **même connexion** entre le builder, l'updater, le deleter, l'auditeur et le composant de maintenance.

Rappel de schéma (§2 de `specification-bdd.md`) : il n'existe **aucune table `dim_*`** — les colonnes catégorielles stockent directement leurs libellés dans `fact_table`.

### Table des matières

0. [Importation des modules](#section_0)
1. [Création des données synthétiques](#section_1)
2. [Construction de la base de données](#section_2)
   - [Connexion et initialisation](#section_2_1)
   - [Construction du schéma](#section_2_2)
   - [Vérification de l'état initial](#section_2_3)
3. [Mise à jour de la base de données](#section_3)
   - [Mise à jour de lignes existantes](#section_3_1)
   - [Ajout de nouvelles observations](#section_3_2)
4. [Suppression d'observations](#section_4)
   - [Suppression filtrée simple](#section_4_1)
   - [Suppression totale d'une modalité](#section_4_2)
5. [Audit de la base de données](#section_5)
   - [Audit basique](#section_5_1)
   - [Audit standard](#section_5_2)
   - [Audit complet](#section_5_3)
6. [Maintenance physique (DuckLake)](#section_6)
   - [Opérations individuelles](#section_6_1)
   - [Maintenance complète](#section_6_2)
7. [Voyage dans le temps (time-travel)](#section_7)
   - [Consultation d'un snapshot par version](#section_7_1)
   - [Consultation d'un snapshot par horodatage](#section_7_2)

## 0. Importation des modules <a id="section_0"></a>

In [ ]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import os
import shutil
import sys
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

# Ajout du chemin racine du projet
sys.path.append("..")

# Connexion DuckLake
from dt_ducklake_manager.connection import DuckLakeConnector

# Maintenance et audit
from dt_ducklake_manager.maintenance import (
    DatabaseAuditor,
    DuckLakeMaintenance,
    MaintenancePolicy,
    ValidationLevel,
)

# Opérations sur les données
from dt_ducklake_manager.operations import DatabaseDeleter, DatabaseUpdater

# Construction du schéma
from dt_ducklake_manager.schema import DuckLakeTablesBuilder

## 1. Création des données synthétiques <a id="section_1"></a>

In [ ]:
# Initialisation du générateur aléatoire pour la reproductibilité
np.random.seed(42)

# Paramètres du jeu de données synthétiques
N_ROWS = 200
CATEGORICAL_THRESHOLD = 8
START_DATE = datetime(2024, 1, 1)

# Modalités des colonnes catégorielles
indicators = ["temperature", "humidity", "pressure", "wind_speed"]
countries = ["France", "Germany", "Italy", "Spain"]
kinds = ["forecast", "observation"]
models = ["model_A", "model_B", "model_C"]
trainings = ["train_v1", "train_v2"]
horizons = [1, 3, 7, 14]

# Construction du DataFrame initial
data_list = []
for i in range(N_ROWS):
    date = START_DATE + timedelta(days=i % 180)
    row = {
        "indicator": np.random.choice(indicators),
        "country": np.random.choice(countries),
        "kind": np.random.choice(kinds),
        "model": np.random.choice(models),
        "training": np.random.choice(trainings),
        "horizon": np.random.choice(horizons),
        "date": date,
        "value": np.random.uniform(10, 100),
        "lower_bound": np.random.uniform(5, 50) if np.random.random() > 0.4 else None,
        "upper_bound": np.random.uniform(50, 150) if np.random.random() > 0.4 else None,
        "quality_score": np.random.uniform(0, 1),
    }
    data_list.append(row)

df_origin = pd.DataFrame(data_list)
df_origin["date"] = pd.to_datetime(df_origin["date"])

# Clé primaire composite
PK_COLUMNS = ["indicator", "country", "kind", "model", "training", "horizon", "date"]
df_origin = df_origin.drop_duplicates(subset=PK_COLUMNS, keep="first").reset_index(
    drop=True
)

# Labels des colonnes (utilisés dans les métadonnées)
LABELS = {
    "indicator": "Indicateur",
    "country": "Pays",
    "kind": "Type",
    "model": "Modèle",
    "training": "Entraînement",
    "horizon": "Horizon",
    "date": "Date",
    "value": "Valeur",
    "lower_bound": "Borne inférieure",
    "upper_bound": "Borne supérieure",
    "quality_score": "Score de qualité",
}

# Chemins du catalogue et des données DuckLake
CATALOG_PATH = os.path.join("../outputs", "workflow_integration.ducklake")
DATA_PATH = os.path.join("../outputs", "workflow_integration_data/")

# Affichage du résumé du jeu de données
print(f"Nombre de lignes après déduplication : {len(df_origin)}")
print(f"Seuil catégoriel : {CATEGORICAL_THRESHOLD}")
print(f"Catalogue : {CATALOG_PATH}")
df_origin.head()

## 2. Construction de la base de données <a id="section_2"></a>

### 2.1. Connexion et initialisation <a id="section_2_1"></a>

In [ ]:
# Suppression d'un éventuel état résiduel pour garantir un départ propre
for suffix in ["", ".wal"]:
    path_to_remove = CATALOG_PATH + suffix
    if os.path.exists(path_to_remove):
        os.remove(path_to_remove)
        print(f"Fichier supprimé : {path_to_remove}")
if os.path.exists(DATA_PATH):
    shutil.rmtree(DATA_PATH)
    print(f"Répertoire supprimé : {DATA_PATH}")

# Ouverture de la connexion DuckLake en lecture-écriture
conn = DuckLakeConnector(CATALOG_PATH, DATA_PATH).connect()
print("Connexion DuckLake ouverte.")

### 2.2. Construction du schéma <a id="section_2_2"></a>

In [ ]:
# Initialisation du builder avec clé primaire composite.
# categorical_threshold ne pilote qu'un indicateur d'interface (metadata.is_categorical) :
# les libellés catégoriels restent stockés directement dans fact_table, il n'existe pas
# de table de dimension.
builder = DuckLakeTablesBuilder(
    df=df_origin,
    categorical_threshold=CATEGORICAL_THRESHOLD,
    primary_keys=PK_COLUMNS,
    connection=conn,
)

# Construction du schéma complet (fact_table + metadata + dataset_metadata).
# run_id/commit_message/commit_info sont acceptés par chaque opération d'écriture et
# enregistrés sur le snapshot DuckLake correspondant — approfondi au notebook 7.
report_build = builder.build_schema(
    column_labels=LABELS,
    run_id="build-2026-workflow",
    commit_message="Construction initiale du jeu de résultats",
)
print(report_build.summary())

# Vérification rapide du nombre de lignes insérées
n = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"Lignes dans fact_table après construction : {n}")

# Snapshot de référence juste après la construction (fact_table existe déjà) — repris
# en Section 7 pour le voyage dans le temps, avant que les mises à jour suivantes ne
# modifient la base.
snapshot_after_build = conn.execute(
    "SELECT max(snapshot_id) FROM ducklake_snapshots('db')"
).fetchone()[0]
print(f"Snapshot de référence (juste après construction) : {snapshot_after_build}")

### 2.3. Vérification de l'état initial <a id="section_2_3"></a>

In [ ]:
# Affichage du schéma (tables créées, colonnes, types)
builder.display_schema()

In [ ]:
# Lecture des métadonnées : statut catégoriel, type SQL, libellé
df_meta = conn.execute("SELECT * FROM metadata").fetchdf()
display(df_meta[["name", "label", "sql_type", "is_categorical"]])

In [ ]:
# Aucune table de dimension : les colonnes catégorielles stockent leurs libellés
# d'origine directement dans fact_table.
all_tables = sorted([row[0] for row in conn.execute("SHOW TABLES").fetchall()])
print(f"Tables du schéma : {all_tables}")

print("\nModalités de 'country', lues directement dans fact_table (pas de jointure) :")
display(
    conn.execute("SELECT DISTINCT country FROM fact_table ORDER BY country").fetchdf()
)

## 3. Mise à jour de la base de données <a id="section_3"></a>

Le `DatabaseUpdater` effectue un **upsert** : les lignes existantes (identifiées par la clé primaire) sont mises à jour, les nouvelles lignes sont insérées.

In [ ]:
# Initialisation du composant de mise à jour
# La connexion est partagée avec le builder — les opérations portent sur le même état
updater = DatabaseUpdater(
    connection=conn,
    categorical_threshold=CATEGORICAL_THRESHOLD,
)

### 3.1. Mise à jour de lignes existantes <a id="section_3_1"></a>

Modification des colonnes numériques (`value`, `quality_score`) de 10 lignes existantes.

In [ ]:
# Échantillon de 10 lignes existantes : les libellés catégoriels sont déjà dans
# fact_table, un simple SELECT suffit (pas de jointure vers une table de dimension).
sample_rows = conn.execute("SELECT * FROM fact_table LIMIT 10").fetchdf()

# Modification des colonnes numériques uniquement
update_31 = sample_rows.copy()
update_31["value"] = update_31["value"] * 1.10  # Augmentation de 10 %
update_31["quality_score"] = update_31["quality_score"] * 0.95  # Légère dégradation

print(f"Lignes à mettre à jour : {len(update_31)}")
display(update_31.head(3))

In [ ]:
# Exécution de la mise à jour 3.1
success_31 = updater.update_database(
    update_df=update_31,
    check_duplicates_db=False,
    check_duplicates_update=False,
    keep="last",
    use_transaction=True,
)
print(f"Mise à jour 3.1 réussie : {success_31}")
print(updater.last_report.summary())

### 3.2. Ajout de nouvelles observations <a id="section_3_2"></a>

Insertion de 20 nouvelles lignes correspondant à une nouvelle fenêtre temporelle (2025).
Ces lignes n'existent pas dans la base : l'upsert les insère sans modifier les lignes existantes.

In [ ]:
# Génération de 20 nouvelles lignes avec des dates en 2025
np.random.seed(10)
new_start_date = datetime(2025, 1, 1)

new_rows = []
for i in range(20):
    new_rows.append(
        {
            "indicator": np.random.choice(indicators),
            "country": np.random.choice(countries),
            "kind": np.random.choice(kinds),
            "model": np.random.choice(models),
            "training": np.random.choice(trainings),
            "horizon": np.random.choice(horizons),
            "date": new_start_date + timedelta(days=i),
            "value": np.random.uniform(10, 100),
            "lower_bound": np.random.uniform(5, 50),
            "upper_bound": np.random.uniform(50, 150),
            "quality_score": np.random.uniform(0.5, 1.0),
        }
    )

df_new = pd.DataFrame(new_rows)
df_new["date"] = pd.to_datetime(df_new["date"])
df_new = df_new.drop_duplicates(subset=PK_COLUMNS, keep="first").reset_index(drop=True)

n_before = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"Lignes dans fact_table avant insertion : {n_before}")
print(f"Nouvelles observations à insérer       : {len(df_new)}")
display(df_new.head(3))

In [ ]:
# Exécution de la mise à jour 3.2 — insertion des nouvelles lignes
success_32 = updater.update_database(
    update_df=df_new,
    check_duplicates_db=True,
    check_duplicates_update=True,
    keep="last",
    use_transaction=True,
)
n_after = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"Mise à jour 3.2 réussie : {success_32}")
print(f"Lignes dans fact_table après insertion : {n_after}  (+{n_after - n_before})")

## 4. Suppression d'observations <a id="section_4"></a>

`DatabaseDeleter.delete_rows()` accepte une liste de filtres `(colonne, opérateur, valeur)` portant directement sur les libellés stockés dans `fact_table` — aucune résolution d'identifiant n'est nécessaire. Il retourne un `OperationReport` (`rows_deleted`, `summary()`). `perform_cleanup=True` supprime, après coup, les colonnes de valeurs devenues entièrement `NULL` du fait de la suppression — il n'existe plus de notion de « modalité orpheline » à nettoyer, faute de table de dimension.

In [ ]:
# Initialisation du composant de suppression
# La connexion est partagée avec l'updater — les opérations portent sur le même état
deleter = DatabaseDeleter(
    connection=conn,
)

### 4.1. Suppression filtrée simple <a id="section_4_1"></a>

Suppression des observations de type `forecast` avec `horizon = 14`.

In [ ]:
n_before_41 = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
n_target_41 = conn.execute(
    "SELECT COUNT(*) FROM fact_table WHERE kind = 'forecast' AND horizon = 14"
).fetchone()[0]
print(f"Lignes cibles (forecast + horizon 14) : {n_target_41}")

In [ ]:
# Exécution de la suppression 4.1
filters_41 = [
    ("kind", "=", "forecast"),
    ("horizon", "=", 14),
]
report_41 = deleter.delete_rows(
    filters=filters_41,
    use_transaction=True,
    perform_cleanup=True,
)
n_after_41 = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"Lignes supprimées : {report_41.rows_deleted}")
print(f"Lignes restantes  : {n_after_41}  (avant : {n_before_41})")

# 'forecast' reste présent dans fact_table : d'autres horizons le portent encore
print("\nModalités de 'kind' encore présentes :")
display(conn.execute("SELECT DISTINCT kind FROM fact_table ORDER BY kind").fetchdf())

### 4.2. Suppression totale d'une modalité <a id="section_4_2"></a>

Suppression de **toutes** les lignes correspondant au pays `Italy` : après la suppression, `'Italy'` n'apparaît plus du tout dans `fact_table.country`.

In [ ]:
print("Répartition de 'country' avant suppression :")
display(
    conn.execute(
        "SELECT country, COUNT(*) AS n FROM fact_table GROUP BY country ORDER BY"
        " country"
    ).fetchdf()
)

n_italy = conn.execute(
    "SELECT COUNT(*) FROM fact_table WHERE country = 'Italy'"
).fetchone()[0]
print(f"\nLignes 'Italy' à supprimer : {n_italy}")

In [ ]:
# Exécution de la suppression 4.2
filters_42 = [("country", "=", "Italy")]
report_42 = deleter.delete_rows(
    filters=filters_42,
    use_transaction=True,
    perform_cleanup=True,
)
print(f"Lignes supprimées : {report_42.rows_deleted}")

print("\nModalités de 'country' après suppression ('Italy' absent) :")
display(
    conn.execute("SELECT DISTINCT country FROM fact_table ORDER BY country").fetchdf()
)

## 5. Audit de la base de données <a id="section_5"></a>

`DatabaseAuditor` valide la cohérence du schéma (`fact_table`, `metadata`, `dataset_metadata`) ainsi que les problèmes de performance potentiels.
Trois niveaux de validation sont disponibles : `BASIC`, `STANDARD`, `COMPREHENSIVE`.

In [ ]:
# Initialisation de l'auditeur avec la connexion partagée
auditor = DatabaseAuditor(
    connection=conn,
)

### 5.1. Audit basique <a id="section_5_1"></a>

Le niveau `BASIC` vérifie uniquement l'existence des tables obligatoires (`fact_table`, `metadata`)  
et la cohérence minimale du schéma. Rapide, adapté aux contrôles de santé en production.

In [ ]:
# Exécution de l'audit basique
report_basic = auditor.validate_database(ValidationLevel.BASIC)

print(f"Niveau d'audit       : {report_basic.validation_level.value}")
print(f"Problèmes détectés   : {len(report_basic.issues)}")
print(f"Problèmes critiques  : {report_basic.get_critical_issues_count()}")
print(f"Résumé               : {report_basic.validation_summary}")

### 5.2. Audit standard <a id="section_5_2"></a>

Le niveau `STANDARD` ajoute la vérification de la cohérence entre `metadata` et `fact_table`
(chaque colonne de la table des faits doit avoir sa ligne de métadonnées, et réciproquement)
ainsi que la cohérence des types de données déclarés.

In [ ]:
# Exécution de l'audit standard
report_standard = auditor.validate_database(ValidationLevel.STANDARD)

print(f"Niveau d'audit       : {report_standard.validation_level.value}")
print(f"Problèmes détectés   : {len(report_standard.issues)}")
print(f"Recommandations      : {report_standard.recommendations}")

# Détail des problèmes détectés
if report_standard.issues:
    print("\nDétail des problèmes :")
    for issue in report_standard.issues:
        print(
            f"  [{issue.severity.value.upper()}] {issue.table_name} —"
            f" {issue.description}"
        )
else:
    print("Aucun problème détecté à ce niveau.")

### 5.3. Audit complet <a id="section_5_3"></a>

Le niveau `COMPREHENSIVE` inclut en plus la vérification des performances (indexes, fragmentation)  
et un contrôle approfondi des données (valeurs nulles sur les clés primaires, cohérence des types).

In [ ]:
# Exécution de l'audit complet
report_comprehensive = auditor.validate_database(ValidationLevel.COMPREHENSIVE)

print(f"Niveau d'audit       : {report_comprehensive.validation_level.value}")
print(f"Tables validées      : {report_comprehensive.tables_validated}")
print(f"Résumé               : {report_comprehensive.validation_summary}")

# Regroupement des problèmes par sévérité pour faciliter la lecture
for severity_label in ["critical", "high", "medium", "low"]:
    count = report_comprehensive.validation_summary.get(f"{severity_label}_issues", 0)
    if count > 0:
        print(f"\nProblèmes [{severity_label.upper()}] ({count}) :")
        from dt_ducklake_manager.maintenance.auditor import IssueSeverity

        sev = IssueSeverity(severity_label)
        for issue in report_comprehensive.get_issues_by_severity(sev):
            print(f"  - {issue.table_name} : {issue.description}")

if not report_comprehensive.issues:
    print("Base de données saine — aucun problème détecté.")

## 6. Maintenance physique (DuckLake) <a id="section_6"></a>

Chaque INSERT / UPDATE / DELETE dans DuckLake produit un petit fichier Parquet ou un fichier de  
tombstones. `DuckLakeMaintenance` permet de consolider ces fichiers pour maintenir des performances  
de lecture optimales.

Comme les autres composants schema-aware, `DuckLakeMaintenance` transporte **l'alias du catalogue
et le schéma ensemble** : `catalog_alias` doit correspondre à celui passé au `DuckLakeConnector`
(défaut `"db"`) et `schema` désigne le jeu de résultats visé (défaut `"main"`).

In [ ]:
# Initialisation du composant de maintenance avec la connexion partagée.
# L'alias du catalogue et le schéma sont fournis explicitement, au même titre : ils
# correspondent ici aux valeurs par défaut du DuckLakeConnector ouvert en Section 2.1.
maint = DuckLakeMaintenance(connection=conn, catalog_alias="db", schema="main")

### 6.1. Opérations individuelles <a id="section_6_1"></a>

Les quatre opérations DuckLake peuvent être exécutées séparément selon le besoin :
- `merge_files` : fusion des petits fichiers Parquet adjacents
- `rewrite_data_files` : suppression physique des tombstones (lignes effacées)
- `expire_snapshots` : expiration des anciens snapshots selon un seuil en jours
- `cleanup_files` : suppression des fichiers Parquet orphelins (non référencés par un snapshot actif)

In [ ]:
# Fusion des petits fichiers Parquet pour réduire le nombre de handles d'I/O
maint.merge_files(schema="main", table="fact_table")
print("merge_files terminé.")

In [ ]:
# Réécriture des fichiers pour supprimer physiquement les tombstones de suppression
maint.rewrite_data_files(schema="main", table="fact_table")
print("rewrite_data_files terminé.")

In [ ]:
# Expiration des snapshots vieux de plus de 0 jours (conserve uniquement le snapshot
# courant)
# En production, on utilise typiquement older_than_days=30 pour conserver un mois
# d'historique
maint.expire_snapshots(schema="main", older_than_days=0)
print("expire_snapshots terminé (older_than_days=0 pour illustration).")

In [ ]:
# Suppression des fichiers Parquet orphelins après expiration des snapshots
maint.cleanup_files(schema="main")
print("cleanup_files terminé.")

### 6.2. Maintenance complète <a id="section_6_2"></a>

`maintain()` lit l'état du stockage (`storage_report`) et n'exécute que les étapes justifiées par la `MaintenancePolicy` : vidange des lignes inlinées, réécriture, fusion, puis expiration et nettoyage des snapshots quand `retention_days` est renseigné.  
C'est la méthode à appeler en production après un lot de mises à jour intensif.

In [ ]:
# Simulation de quelques opérations supplémentaires pour accumuler des fichiers delta
np.random.seed(99)
extra_rows = df_new.head(5).copy()
extra_rows["value"] = extra_rows["value"] + 5
updater.update_database(
    extra_rows, check_duplicates_db=False, check_duplicates_update=False
)
print("Mise à jour intermédiaire effectuée — fichiers delta générés.")

# Maintenance planifiée avec conservation de 30 jours d'historique
report_maint = maint.maintain(
    MaintenancePolicy(retention_days=30), table="fact_table", schema="main"
)
print(report_maint.summary())

## 7. Voyage dans le temps (time-travel) <a id="section_7"></a>

DuckLake conserve un historique complet des snapshots. Le time-travel s'effectue via la  
clause SQL `AT` sur la **connexion existante**, car DuckLake verrouille le fichier catalogue  
au niveau processus et interdit deux connexions simultanées sur le même fichier.  
`DuckLakeConnector.at_clause()` génère la clause `AT (VERSION => n)` ou `AT (TIMESTAMP => ...)`  
à insérer directement dans les requêtes `FROM`.

Le snapshot de référence (`snapshot_after_build`) a été capturé juste après la construction du
schéma (Section 2.2), donc **après** la création de `fact_table` — contrairement au tout premier
snapshot du catalogue (`snapshot_id = 0`), qui la précède et ne peut pas être interrogé via `AT`.

> **Note** : les snapshots expirés en Section 6.1 ne sont plus accessibles par time-travel.  
> `snapshot_after_build` n'est donc disponible ci-dessous que si `older_than_days=0` n'a pas  
> déjà purgé l'historique complet. L'exemple reste illustratif.

### 7.1. Consultation d'un snapshot par version <a id="section_7_1"></a>

In [ ]:
# Liste des snapshots disponibles dans le catalogue
df_snapshots = conn.execute(
    "SELECT snapshot_id, snapshot_time FROM ducklake_snapshots('db') ORDER BY"
    " snapshot_id"
).fetchdf()
print(f"Snapshots disponibles : {len(df_snapshots)}")
display(df_snapshots)

In [ ]:
# Le snapshot de référence a déjà été capturé en Section 2.2 (snapshot_after_build),
# juste après la construction du schéma — rappel de sa valeur ici.
print(f"Snapshot de référence pour le voyage dans le temps : {snapshot_after_build}")

In [ ]:
# Time-travel par numéro de version via la clause AT sur la connexion courante
# DuckLake interdit d'attacher le même fichier catalogue deux fois dans le même
# processus :
# at_clause() génère la clause SQL AT (VERSION => n) à utiliser directement dans FROM.
if snapshot_after_build in df_snapshots["snapshot_id"].values:
    connector_past = DuckLakeConnector(
        CATALOG_PATH,
        DATA_PATH,
        snapshot_version=snapshot_after_build,
    )
    at = connector_past.at_clause()

    n_past = conn.execute(f"SELECT COUNT(*) FROM fact_table {at}").fetchone()[0]
    n_current = conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]

    print(
        f"Lignes dans le snapshot {snapshot_after_build} (juste après construction) :"
        f" {n_past}"
    )
    print(
        f"Lignes dans la version courante                                        :"
        f" {n_current}"
    )
    print(
        f"Différence                                                             :"
        f" {n_current - n_past}"
    )
else:
    print(
        f"Le snapshot {snapshot_after_build} a été expiré à l'étape 6.1."
        "\nPour préserver l'historique, utiliser older_than_days > 0 dans"
        " expire_snapshots()."
    )

### 7.2. Consultation d'un snapshot par horodatage <a id="section_7_2"></a>

`snapshot_time` accepte une chaîne ISO-8601 et ouvre le snapshot le plus récent  
antérieur ou égal à l'horodatage fourni. Utile pour auditer l'état de la base  
à une date précise sans avoir à connaître le numéro de version.

In [ ]:
# Time-travel par horodatage via la clause AT sur la connexion courante
# On utilise l'horodatage du snapshot de référence (snapshot_after_build) plutôt que
# le tout premier snapshot (id=0), qui précède la création de fact_table.
if not df_snapshots.empty:
    ref_row = df_snapshots[df_snapshots["snapshot_id"] == snapshot_after_build]
    if ref_row.empty:
        print(
            f"Le snapshot {snapshot_after_build} a été expiré à l'étape 6.1."
            "Pour préserver l'historique, utiliser older_than_days > 0 dans"
            " expire_snapshots()."
        )
    else:
        snapshot_time_str = str(ref_row.iloc[0]["snapshot_time"])
        print(
            f"Horodatage du snapshot de référence ({snapshot_after_build}) :"
            f" {snapshot_time_str}"
        )

        connector_time = DuckLakeConnector(
            CATALOG_PATH,
            DATA_PATH,
            snapshot_time=snapshot_time_str,
        )
        at = connector_time.at_clause()

        try:
            n_time = conn.execute(
                f"SELECT COUNT(*) FROM main.fact_table {at}"
            ).fetchone()[0]
            print(f"Lignes dans le snapshot à {snapshot_time_str} : {n_time}")
        except Exception as e:
            print(f"Time-travel par horodatage indisponible : {e}")
else:
    print("Aucun snapshot disponible dans le catalogue.")

In [ ]:
# Fermeture propre de la connexion principale
conn.close()
print("Connexion principale fermée.")